In [1]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 1s (9,700 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 121703 files and dire

#데이터 불러오기

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='NanumBarunGothic')

!pip install tspoon
import tspoon as tsp

In [3]:
!pip install xmltodict
import pandas as pd

import requests
import xml.etree.ElementTree as ET
import xml.dom.minidom
import xmltodict

import json
from urllib.request import urlopen

def getECOS(topic,period,beg,end,item1='?',item2='?',item3='?',item4='?', df=None):
    key = ''
    url = 'https://ecos.bok.or.kr/api/StatisticSearch/'+'OIT9OYFOZDZWLHSBQJWU'+'/xml/kr/'+str(1)+'/'+str(50000)+'/'+topic+'/'+period+'/'+beg+'/'+end \
            +'/'+item1+'/'+item2+'/'+item3+'/'+item4
    # print(url)
    ## call OpenAPI URL
    response = requests.get(url)

    ## get API return value upon the http request
    if response.status_code == 200:
        try:
            contents = response.text
            ecosRoot = ET.fromstring(contents)

            # error check
            if ecosRoot[0].text[:4] in ("INFO","ERRO"):
                print(ecosRoot[0].text + " : " + ecosRoot[1].text)

            else:
                #print(contents)
                #dom = xml.dom.minidom.parse(xml_fname)
                dom = xml.dom.minidom.parseString(contents)
                pretty_xml_as_string = dom.toprettyxml(indent=" ")
                # print(pretty_xml_as_string)
                dic = xmltodict.parse(pretty_xml_as_string)

                n = int(dic['StatisticSearch']['list_total_count']['#text'])
                df_ecos = pd.DataFrame(index = [dic['StatisticSearch']['row'][i]['TIME'] for i in range(n)])
                df_ecos[dic['StatisticSearch']['row'][0]['ITEM_NAME1']] = [float(dic['StatisticSearch']['row'][i]['DATA_VALUE']) for i in range(n)]

                if type(df)==type(pd.DataFrame()):
                    df_ecos = df.merge(df_ecos, left_index=True, right_index=True, how='left')

                return df_ecos

        except Exception as e:
            print(str(e))


def getKOSIS(table,period,beg,end,item, orgId='101', obj1='',obj2='',obj3='',obj4='',obj5='',obj6='',obj7='',obj8='', title=None, title_no='0', debug=False):
     # Korean Statistics
    key = 'MzhiMmQwZTYyMWZiNGM1ZGFkZDJmZTI2OTE1OGU2NzQ='
    url = 'https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey='+key \
            +'&orgId='+orgId\
            +'&tblId='+table \
            +'&prdSe='+period \
            +'&startPrdDe='+beg \
            +'&endPrdDe='+end \
            +'&itmId='+item \
            +'&objL1='+obj1 \
            +'&objL2='+obj2 \
            +'&objL3='+obj3 \
            +'&objL4='+obj4 \
            +'&objL5='+obj5 \
            +'&objL6='+obj6 \
            +'&objL7='+obj7 \
            +'&objL8='+obj8 \
            +'&format=json&jsonVD=Y&outputFields=ITM_NM+PRD_DE+DT'
    res = requests.get(url)
    py_json = res.json()

    data = []
    for v in py_json:
        value = []
        value.append(v['PRD_DE'])

        # ✅ DT 값이 '-' 또는 ''일 때 NaN으로 처리
        try:
            value.append(float(v['DT']))
        except (ValueError, KeyError):
            value.append(np.nan)

        data.append(value)

    df = pd.DataFrame(data, columns=['PRD_DE', 'DT'])

    if debug:
        print(df.head())

    return df


    #get json data from url
    with urlopen(url) as url_:
        json_file = url_.read()

    py_json = json.loads(json_file.decode('utf-8'))


    #data transformation
    data = []

    for i, v in enumerate(py_json):
        value = []
        value.append(v['PRD_DE'])
        value.append(float(v['DT']))

        data.append(value)

    if title is not None:
        title = str(title)
    elif title_no=='0':
        title = v['ITM_NM_ENG']
    else:
        title = v['C'+title_no+'_NM_ENG']

    df_kosis = pd.DataFrame({v['PRD_SE']:[x[0] for x in data], title:[x[1] for x in data]}).set_index(v['PRD_SE'])

    if debug:
        return v

    return df_kosis

In [4]:
import pandas as pd
import numpy as np
import os
import seaborn as sns

In [5]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [6]:
# change directory to where you have your data file!!
%cd gdrive/MyDrive/GDP/데이터

/content/gdrive/MyDrive/GDP/데이터


##GDP

In [7]:
gdp = getECOS('200Y104', 'Q', '2014Q1', '2025Q2','110601').reset_index()
gdp

,index,도소매업
0,2014Q1,37386.7
1,2014Q2,37479.5
2,2014Q3,37280.3
3,2014Q4,37002.5
4,2015Q1,38208.1
5,2015Q2,38369.3
6,2015Q3,38844.2
7,2015Q4,39170.8
8,2016Q1,39397.0
9,2016Q2,39598.1


In [8]:
gdp = gdp.rename(columns={'도소매업':'gdp'})

In [9]:
gdp

,index,gdp
0,2014Q1,37386.7
1,2014Q2,37479.5
2,2014Q3,37280.3
3,2014Q4,37002.5
4,2015Q1,38208.1
5,2015Q2,38369.3
6,2015Q3,38844.2
7,2015Q4,39170.8
8,2016Q1,39397.0
9,2016Q2,39598.1


##bsi

In [10]:
업황실적BSI = getECOS('512Y007','M','201401','202506','AA','G4500').reset_index()
매출실적BSI = getECOS('512Y007','M','201401','202506','AB','G4500').reset_index()
채산성실적BSI = getECOS('512Y007','M','201401','202506','AE','G4500').reset_index()
자금사정실적BSI = getECOS('512Y007','M','201401','202506','AO','G4500').reset_index()
인력사정실적BSI = getECOS('512Y007','M','201401','202506','AJ','G4500').reset_index()

업황실적BSI = 업황실적BSI.rename(columns={'업황실적BSI 1)':'업황실적BSI'})
매출실적BSI = 매출실적BSI.rename(columns={'매출실적BSI 2)':'매출실적BSI'})
채산성실적BSI = 채산성실적BSI.rename(columns={'채산성실적BSI 6)':'채산성실적BSI'})
자금사정실적BSI = 자금사정실적BSI.rename(columns={'자금사정실적BSI 6)':'자금사정실적BSI'})
인력사정실적BSI = 인력사정실적BSI.rename(columns={'인력사정실적BSI 3)':'인력사정실적BSI'})

In [11]:
인력사정실적BSI

,index,인력사정실적BSI
0,201401,87.0
1,201402,91.0
2,201403,95.0
3,201404,93.0
4,201405,92.0
...,...,...
133,202502,82.0
134,202503,81.0
135,202504,78.0
136,202505,84.0


In [12]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [업황실적BSI, 매출실적BSI, 채산성실적BSI, 자금사정실적BSI, 인력사정실적BSI]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
df_merged = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
df_merged

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI
0,201401,71.0,76.0,83.0,85.0,87.0
1,201402,71.0,79.0,79.0,85.0,91.0
2,201403,65.0,80.0,78.0,86.0,95.0
3,201404,75.0,85.0,83.0,90.0,93.0
4,201405,68.0,81.0,81.0,87.0,92.0
...,...,...,...,...,...,...
133,202502,61.0,72.0,77.0,79.0,82.0
134,202503,60.0,65.0,70.0,77.0,81.0
135,202504,61.0,69.0,73.0,78.0,78.0
136,202505,61.0,73.0,71.0,76.0,84.0


In [13]:
df_merged.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
채산성실적BSI,0
자금사정실적BSI,0
인력사정실적BSI,0


In [14]:
import pandas as pd

# index가 숫자라면 문자열로 변환
df_merged['index'] = df_merged['index'].astype(str)

# 연도와 월 분리
df_merged['연도'] = df_merged['index'].str[:4].astype(int)
df_merged['월'] = df_merged['index'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
df_merged['분기'] = df_merged['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
df_merged['index'] = df_merged['연도'].astype(str) + df_merged['분기']

# 분기별 평균 집계
df_quarter = df_merged.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(df_quarter.head())

    index    업황실적BSI    매출실적BSI   채산성실적BSI  자금사정실적BSI  인력사정실적BSI      연도     월
0  2014Q1  69.000000  78.333333  80.000000  85.333333  91.000000  2014.0   2.0
1  2014Q2  69.666667  81.666667  81.666667  87.000000  92.333333  2014.0   5.0
2  2014Q3  67.666667  79.000000  80.000000  87.333333  89.333333  2014.0   8.0
3  2014Q4  64.000000  76.666667  78.666667  86.333333  88.666667  2014.0  11.0
4  2015Q1  66.666667  77.333333  80.333333  88.000000  90.333333  2015.0   2.0


In [15]:
df_quarter.head()

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,연도,월
0,2014Q1,69.000000,78.333333,80.000000,85.333333,91.000000,2014.0,2.0
1,2014Q2,69.666667,81.666667,81.666667,87.000000,92.333333,2014.0,5.0
2,2014Q3,67.666667,79.000000,80.000000,87.333333,89.333333,2014.0,8.0
3,2014Q4,64.000000,76.666667,78.666667,86.333333,88.666667,2014.0,11.0
4,2015Q1,66.666667,77.333333,80.333333,88.000000,90.333333,2015.0,2.0


In [16]:
df_quarter = df_quarter.drop(columns=['연도', '월'], errors='ignore')


In [17]:
dfs = [gdp]
df_final = df_quarter.copy()
key_col = 'index'
for df in dfs:
    df_final = df_final.merge(df, on=key_col, how='left')

df_final

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,gdp
0,2014Q1,69.000000,78.333333,80.000000,85.333333,91.000000,37386.7
1,2014Q2,69.666667,81.666667,81.666667,87.000000,92.333333,37479.5
2,2014Q3,67.666667,79.000000,80.000000,87.333333,89.333333,37280.3
3,2014Q4,64.000000,76.666667,78.666667,86.333333,88.666667,37002.5
4,2015Q1,66.666667,77.333333,80.333333,88.000000,90.333333,38208.1
5,2015Q2,68.666667,77.333333,85.666667,89.666667,89.000000,38369.3
6,2015Q3,66.666667,79.666667,83.666667,83.666667,90.333333,38844.2
7,2015Q4,69.666667,81.000000,83.333333,85.000000,87.333333,39170.8
8,2016Q1,63.666667,74.333333,81.000000,85.000000,88.666667,39397.0
9,2016Q2,70.333333,79.000000,82.333333,82.000000,87.333333,39598.1


In [18]:
df_final.to_csv("도소매업1차.csv")

## 생산지수

In [19]:
생산지수 = getKOSIS('DT_1KC2020','Q','201401','202502','T3','101','G')

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산지수['index']=생산지수['PRD_DE'].apply(to_quarter)
생산지수 = 생산지수.drop(columns=["PRD_DE"])
생산지수 = 생산지수.rename(columns={'DT': '생산지수'})
생산지수

,생산지수,index
0,96.9,2014Q1
1,96.5,2014Q2
2,97.1,2014Q3
3,96.0,2014Q4
4,97.5,2015Q1
5,97.5,2015Q2
6,98.3,2015Q3
7,99.8,2015Q4
8,99.3,2016Q1
9,101.3,2016Q2


https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=&itmId=T3+&objL1=G0+&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Q&startPrdDe=201401&endPrdDe=202502&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=101&tblId=DT_1K41012

## 소매판매액지수

In [20]:
소매판매액지수 = getKOSIS('DT_1K41012','Q','201401','202502','T3','101','G0')

In [21]:
소매판매액지수

,PRD_DE,DT
0,201401,85.7
1,201402,85.0
2,201403,86.1
3,201404,86.3
4,201501,87.7
5,201502,87.6
6,201503,88.2
7,201504,91.1
8,201601,91.2
9,201602,92.5


In [22]:
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

소매판매액지수['index']=소매판매액지수['PRD_DE'].apply(to_quarter)
소매판매액지수 = 소매판매액지수.drop(columns=["PRD_DE"])
소매판매액지수 = 소매판매액지수.rename(columns={'DT': '소매판매액지수'})
소매판매액지수

,소매판매액지수,index
0,85.7,2014Q1
1,85.0,2014Q2
2,86.1,2014Q3
3,86.3,2014Q4
4,87.7,2015Q1
5,87.6,2015Q2
6,88.2,2015Q3
7,91.1,2015Q4
8,91.2,2016Q1
9,92.5,2016Q2


## 기업데이터

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
df = pd.read_csv('임지오기업.csv')

In [25]:
도소매업 = df[df['매핑한 산업'] == "도소매업"]

In [26]:
도소매업

,Unnamed: 0,업체코드,종목코드,종목명,업체명,사업자번호,상장일,결산월,산업세세분류,조사일,EBITDA,인건비,"유,무형,리스자산의증가",매출원가,수익(매출액),매핑한 산업
102,204,N320684,A000050,경방,(주)경방,107-81-05232,1956-03-03,12.0,G47119.기타 대형 종합 소매업,2000-01-01,0.000000e+00,0.0,0.000000e+00,4.045600e+10,4.841915e+10,도소매업
103,205,N320684,A000050,경방,(주)경방,107-81-05232,1956-03-03,12.0,G47119.기타 대형 종합 소매업,2000-04-01,0.000000e+00,21443723.0,0.000000e+00,4.162628e+10,4.677861e+10,도소매업
104,206,N320684,A000050,경방,(주)경방,107-81-05232,1956-03-03,12.0,G47119.기타 대형 종합 소매업,2000-07-01,0.000000e+00,-21443723.0,0.000000e+00,3.428558e+10,3.690165e+10,도소매업
105,207,N320684,A000050,경방,(주)경방,107-81-05232,1956-03-03,12.0,G47119.기타 대형 종합 소매업,2000-10-01,1.539412e+10,36379634.0,2.729970e+09,3.598126e+10,4.041373e+10,도소매업
106,208,N320684,A000050,경방,(주)경방,107-81-05232,1956-03-03,12.0,G47119.기타 대형 종합 소매업,2001-01-01,0.000000e+00,0.0,0.000000e+00,4.016496e+10,4.563215e+10,도소매업
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71497,373927,NJB0688,A950170,JTC,(주)제이티씨,NaN,2018-04-06,2.0,G47130.면세점,2024-04-01,1.109332e+10,5284616.0,3.226334e+09,1.960907e+10,7.749555e+10,도소매업
71498,373928,NJB0688,A950170,JTC,(주)제이티씨,NaN,2018-04-06,2.0,G47130.면세점,2024-07-01,1.145466e+10,5532262.0,1.890192e+09,1.798675e+10,7.447296e+10,도소매업
71499,373929,NJB0688,A950170,JTC,(주)제이티씨,NaN,2018-04-06,2.0,G47130.면세점,2024-10-01,-6.333496e+09,-3883023.0,-4.152853e+09,-2.321521e+10,-8.668787e+10,도소매업
71500,373930,NJB0688,A950170,JTC,(주)제이티씨,NaN,2018-04-06,2.0,G47130.면세점,2025-01-01,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,도소매업


In [27]:
columns = ['EBITDA', '인건비']
grouped = 도소매업.groupby('조사일')[columns].sum()
grouped

,EBITDA,인건비
조사일,,
2000-01-01,0.000000e+00,1.551425e+08
2000-04-01,0.000000e+00,3.192892e+08
2000-07-01,0.000000e+00,1.506984e+08
2000-10-01,2.046866e+12,6.233153e+08
2001-01-01,0.000000e+00,2.173980e+08
...,...,...
2024-04-01,2.686747e+12,1.826039e+09
2024-07-01,2.467602e+12,1.805518e+09
2024-10-01,2.546659e+12,1.514847e+09


In [28]:
grouped = grouped.reset_index()

In [29]:
grouped['조사일']=pd.to_datetime(grouped['조사일'])

# 분기(Quarter) 단위로 변환
grouped["index"] = grouped["조사일"].dt.to_period("Q")
grouped = grouped.drop(columns=["조사일"])
print(grouped)

grouped

           EBITDA           인건비   index
0    0.000000e+00  1.551425e+08  2000Q1
1    0.000000e+00  3.192892e+08  2000Q2
2    0.000000e+00  1.506984e+08  2000Q3
3    2.046866e+12  6.233153e+08  2000Q4
4    0.000000e+00  2.173980e+08  2001Q1
..            ...           ...     ...
97   2.686747e+12  1.826039e+09  2024Q2
98   2.467602e+12  1.805518e+09  2024Q3
99   2.546659e+12  1.514847e+09  2024Q4
100  2.697474e+12  1.098011e+09  2025Q1
101  2.276137e+12 -1.098011e+09  2025Q2

[102 rows x 3 columns]


,EBITDA,인건비,index
0,0.000000e+00,1.551425e+08,2000Q1
1,0.000000e+00,3.192892e+08,2000Q2
2,0.000000e+00,1.506984e+08,2000Q3
3,2.046866e+12,6.233153e+08,2000Q4
4,0.000000e+00,2.173980e+08,2001Q1
...,...,...,...
97,2.686747e+12,1.826039e+09,2024Q2
98,2.467602e+12,1.805518e+09,2024Q3
99,2.546659e+12,1.514847e+09,2024Q4
100,2.697474e+12,1.098011e+09,2025Q1


In [30]:
mask = (grouped["index"] >= pd.Period("2014Q1")) & (grouped["index"] <= pd.Period("2025Q2"))
기업= grouped.loc[mask]
기업
기업['합산'] = 기업['EBITDA']+기업['인건비']
기업

/tmp/ipython-input-1389489221.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  기업['합산'] = 기업['EBITDA']+기업['인건비']


,EBITDA,인건비,index,합산
56,1.518341e+12,1.162281e+09,2014Q1,1.519503e+12
57,1.627542e+12,1.168903e+09,2014Q2,1.628711e+12
58,1.703508e+12,1.176945e+09,2014Q3,1.704685e+12
59,1.596549e+12,1.485859e+09,2014Q4,1.598035e+12
60,1.539522e+12,1.206004e+09,2015Q1,1.540728e+12
61,1.415336e+12,1.194008e+09,2015Q2,1.416530e+12
62,1.564392e+12,1.283336e+09,2015Q3,1.565675e+12
63,1.510040e+12,1.661138e+09,2015Q4,1.511701e+12
64,1.210099e+12,1.450101e+09,2016Q1,1.211549e+12
65,1.453357e+12,1.359343e+09,2016Q2,1.454716e+12


##전체 데이터 합산

In [31]:
from functools import reduce
import pandas as pd

# 모든 데이터프레임의 index 컬럼을 문자열로 변환
for df in [df_final,생산지수, 소매판매액지수, 기업]:
    df['index'] = df['index'].astype(str)

# 리스트에 넣고 합치기
dfs = [df_final,생산지수, 소매판매액지수, 기업]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='index', how='outer'), dfs)

/tmp/ipython-input-3800898491.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['index'] = df['index'].astype(str)


In [32]:
df_merged

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,gdp,생산지수,소매판매액지수,EBITDA,인건비,합산
0,2014Q1,69.000000,78.333333,80.000000,85.333333,91.000000,37386.7,96.9,85.7,1.518341e+12,1.162281e+09,1.519503e+12
1,2014Q2,69.666667,81.666667,81.666667,87.000000,92.333333,37479.5,96.5,85.0,1.627542e+12,1.168903e+09,1.628711e+12
2,2014Q3,67.666667,79.000000,80.000000,87.333333,89.333333,37280.3,97.1,86.1,1.703508e+12,1.176945e+09,1.704685e+12
3,2014Q4,64.000000,76.666667,78.666667,86.333333,88.666667,37002.5,96.0,86.3,1.596549e+12,1.485859e+09,1.598035e+12
4,2015Q1,66.666667,77.333333,80.333333,88.000000,90.333333,38208.1,97.5,87.7,1.539522e+12,1.206004e+09,1.540728e+12
5,2015Q2,68.666667,77.333333,85.666667,89.666667,89.000000,38369.3,97.5,87.6,1.415336e+12,1.194008e+09,1.416530e+12
6,2015Q3,66.666667,79.666667,83.666667,83.666667,90.333333,38844.2,98.3,88.2,1.564392e+12,1.283336e+09,1.565675e+12
7,2015Q4,69.666667,81.000000,83.333333,85.000000,87.333333,39170.8,99.8,91.1,1.510040e+12,1.661138e+09,1.511701e+12
8,2016Q1,63.666667,74.333333,81.000000,85.000000,88.666667,39397.0,99.3,91.2,1.210099e+12,1.450101e+09,1.211549e+12
9,2016Q2,70.333333,79.000000,82.333333,82.000000,87.333333,39598.1,101.3,92.5,1.453357e+12,1.359343e+09,1.454716e+12


In [33]:
df_merged.to_csv('도소매업.csv')

# 데이터qoq 반환

In [34]:
import pandas as pd

# df는 위의 데이터프레임
df_qoq = df_merged.copy()

# 분기 순서대로 정렬 (혹시 index가 문자열일 경우를 대비)
df_qoq = df_qoq.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_qoq.select_dtypes(include=['number']).columns

# 전분기 대비 증감률 (%) 계산
df_qoq[num_cols] = df_qoq[num_cols].pct_change() * 100

# 결과 확인
print(df_qoq.head())

    index   업황실적BSI   매출실적BSI  채산성실적BSI  자금사정실적BSI  인력사정실적BSI       gdp  \
0  2014Q1       NaN       NaN       NaN        NaN        NaN       NaN   
1  2014Q2  0.966184  4.255319  2.083333   1.953125   1.465201  0.248217   
2  2014Q3 -2.870813 -3.265306 -2.040816   0.383142  -3.249097 -0.531491   
3  2014Q4 -5.418719 -2.953586 -1.666667  -1.145038  -0.746269 -0.745166   
4  2015Q1  4.166667  0.869565  2.118644   1.930502   1.879699  3.258158   

       생산지수   소매판매액지수    EBITDA        인건비        합산  
0       NaN       NaN       NaN        NaN       NaN  
1 -0.412797 -0.816803  7.192149   0.569748  7.187083  
2  0.621762  1.294118  4.667524   0.687960  4.664668  
3 -1.132853  0.232288 -6.278765  26.247116 -6.256309  
4  1.562500  1.622248 -3.571927 -18.834546 -3.586118  


In [35]:
df_qoq
df_qoq = df_qoq.drop(index=[0, 1, 2,3])

In [36]:
df_qoq

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,gdp,생산지수,소매판매액지수,EBITDA,인건비,합산
4,2015Q1,4.166667,0.869565,2.118644,1.930502,1.879699,3.258158,1.562500,1.622248,-3.571927,-18.834546,-3.586118
5,2015Q2,3.000000,0.000000,6.639004,1.893939,-1.476015,0.421900,0.000000,-0.114025,-8.066513,-0.994704,-8.060978
6,2015Q3,-2.912621,3.017241,-2.334630,-6.691450,1.498127,1.237708,0.820513,0.684932,10.531486,7.481317,10.528915
7,2015Q4,4.500000,1.673640,-0.398406,1.593625,-3.321033,0.840795,1.525941,3.287982,-3.474332,29.439090,-3.447354
8,2016Q1,-8.612440,-8.230453,-2.800000,0.000000,1.526718,0.577471,-0.501002,0.109769,-19.863079,-12.704340,-19.855212
9,2016Q2,10.471204,6.278027,1.646091,-3.529412,-1.503759,0.510445,2.014099,1.425439,20.102284,-6.258746,20.070732
10,2016Q3,0.000000,-2.531646,-1.214575,4.065041,0.381679,-0.128542,-0.789733,-1.513514,20.635901,4.432232,20.620760
11,2016Q4,1.895735,6.493506,2.049180,-2.343750,1.901141,0.883248,0.597015,2.195390,8.364802,17.397561,8.372109
12,2017Q1,6.046512,5.691057,1.606426,0.400000,-0.746269,-0.740917,-0.989120,-0.107411,-13.430523,-14.378691,-13.431354
13,2017Q2,-0.877193,1.153846,0.790514,1.992032,-5.263158,2.502468,1.198801,0.537634,-3.543691,-1.537073,-3.541952


In [37]:
df_qoq.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
채산성실적BSI,0
자금사정실적BSI,0
인력사정실적BSI,0
gdp,0
생산지수,0
소매판매액지수,0
EBITDA,0


In [38]:
df_qoq.to_csv('도소매업qoq.csv')

# 데이터yoy 반환

In [39]:
import pandas as pd

# df_merged는 원본 데이터프레임
df_yoy = df_merged.copy()

# 분기 순서대로 정렬
df_yoy = df_yoy.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_yoy.select_dtypes(include=['number']).columns

# 전년 동분기 대비 증감률 (%) 계산 (4분기 차이)
df_yoy[num_cols] = df_yoy[num_cols].pct_change(periods=4) * 100

# 결과 확인
print(df_yoy.head(10))

    index   업황실적BSI   매출실적BSI  채산성실적BSI  자금사정실적BSI  인력사정실적BSI       gdp  \
0  2014Q1       NaN       NaN       NaN        NaN        NaN       NaN   
1  2014Q2       NaN       NaN       NaN        NaN        NaN       NaN   
2  2014Q3       NaN       NaN       NaN        NaN        NaN       NaN   
3  2014Q4       NaN       NaN       NaN        NaN        NaN       NaN   
4  2015Q1 -3.381643 -1.276596  0.416667   3.125000  -0.732601  2.197038   
5  2015Q2 -1.435407 -5.306122  4.897959   3.065134  -3.610108  2.374098   
6  2015Q3 -1.477833  0.843882  4.583333  -4.198473   1.119403  4.194977   
7  2015Q4  8.854167  5.652174  5.932203  -1.544402  -1.503759  5.859874   
8  2016Q1 -4.500000 -3.879310  0.829876  -3.409091  -1.845018  3.111644   
9  2016Q2  2.427184  2.155172 -3.891051  -8.550186  -1.872659  3.202560   

       생산지수   소매판매액지수     EBITDA        인건비         합산  
0       NaN       NaN        NaN        NaN        NaN  
1       NaN       NaN        NaN        NaN        NaN  
2  

In [40]:
df_yoy = df_yoy.dropna()

In [41]:
df_yoy

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,gdp,생산지수,소매판매액지수,EBITDA,인건비,합산
4,2015Q1,-3.381643,-1.276596,0.416667,3.125000,-0.732601,2.197038,0.619195,2.333722,1.394972,3.761821,1.396782
5,2015Q2,-1.435407,-5.306122,4.897959,3.065134,-3.610108,2.374098,1.036269,3.058824,-13.038470,2.147714,-13.027571
6,2015Q3,-1.477833,0.843882,4.583333,-4.198473,1.119403,4.194977,1.235839,2.439024,-8.166480,9.039560,-8.154601
7,2015Q4,8.854167,5.652174,5.932203,-1.544402,-1.503759,5.859874,3.958333,5.561993,-5.418533,11.796467,-5.402527
8,2016Q1,-4.500000,-3.879310,0.829876,-3.409091,-1.845018,3.111644,1.846154,3.990878,-21.397708,20.240150,-21.365116
9,2016Q2,2.427184,2.155172,-3.891051,-8.550186,-1.872659,3.202560,3.897436,5.593607,2.686357,13.847066,2.695764
10,2016Q3,5.500000,-3.347280,-2.788845,1.992032,-2.952030,1.809794,2.238047,3.287982,12.073596,10.617393,12.072403
11,2016Q4,2.870813,1.234568,-0.400000,-1.960784,2.290076,1.852656,1.302605,2.195390,25.819725,0.326819,25.791712
12,2017Q1,19.371728,16.591928,4.115226,-1.568627,0.000000,0.517552,0.805639,1.973684,35.919219,-1.597472,35.874315
13,2017Q2,7.109005,10.970464,3.238866,4.065041,-3.816794,2.509716,0.000000,1.081081,9.159175,3.358986,9.153755


In [42]:
df_yoy.to_csv('도소매업yoy.csv')

# 상관계수 보기(qoq)

In [43]:
qoq = pd.read_csv('도소매업qoq.csv')
qoq = qoq.drop(columns=["Unnamed: 0"])

In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# GDP_QoQ 기준
target = 'gdp'

In [45]:
# 수치형 컬럼만 선택
num_cols = qoq.select_dtypes(include=['number']).columns

# GDP 기준 상관계수 계산
corr = qoq[num_cols].corr()[target].sort_values(ascending=False)

print("📊 GDP 대비 단순 상관계수")
print(corr)

📊 GDP 대비 단순 상관계수
gdp          1.000000
생산지수         0.446011
업황실적BSI      0.306582
매출실적BSI      0.291377
합산           0.172554
EBITDA       0.171846
소매판매액지수      0.072577
채산성실적BSI     0.071340
인건비          0.017511
인력사정실적BSI    0.005657
자금사정실적BSI   -0.006008
Name: gdp, dtype: float64


In [46]:
# df2 사본
df2 = qoq.copy()

# target 및 feature 지정
target_col = 'gdp'
feature_cols = [col for col in df2.columns if col not in ['index', target_col]]

# 결과 저장용 데이터프레임
corr_df = pd.DataFrame(columns=['Feature', 'Lag', 'Correlation'])

# lag 1~4 생성 및 상관계수 계산
for col in feature_cols:
    for lag in range(1, 5):  # lag 1~4
        lag_col_name = f"{col}_lag{lag}"
        df2[lag_col_name] = df2[col].shift(lag)  # 실제 데이터에 lag 컬럼 추가
        corr = df2[target_col].corr(df2[lag_col_name])  # 상관계수 계산
        corr_df = pd.concat(
            [corr_df, pd.DataFrame({'Feature': [col], 'Lag': [lag], 'Correlation': [corr]})],
            ignore_index=True
        )

# 상관계수 높은 순으로 정렬
corr_df.sort_values(by='Correlation', ascending=False, inplace=True)
corr_df.reset_index(drop=True, inplace=True)

corr_df

/tmp/ipython-input-1052179002.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  corr_df = pd.concat(


,Feature,Lag,Correlation
0,인건비,4,0.398273
1,자금사정실적BSI,4,0.307693
2,소매판매액지수,4,0.306264
3,생산지수,4,0.279582
4,채산성실적BSI,3,0.278718
5,업황실적BSI,4,0.276083
6,인력사정실적BSI,3,0.269025
7,매출실적BSI,4,0.238167
8,소매판매액지수,1,0.205858
9,자금사정실적BSI,3,0.177724


In [47]:
df2

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,gdp,생산지수,소매판매액지수,EBITDA,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
0,2015Q1,4.166667,0.869565,2.118644,1.930502,1.879699,3.258158,1.562500,1.622248,-3.571927,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015Q2,3.000000,0.000000,6.639004,1.893939,-1.476015,0.421900,0.000000,-0.114025,-8.066513,...,NaN,NaN,-18.834546,NaN,NaN,NaN,-3.586118,NaN,NaN,NaN
2,2015Q3,-2.912621,3.017241,-2.334630,-6.691450,1.498127,1.237708,0.820513,0.684932,10.531486,...,NaN,NaN,-0.994704,-18.834546,NaN,NaN,-8.060978,-3.586118,NaN,NaN
3,2015Q4,4.500000,1.673640,-0.398406,1.593625,-3.321033,0.840795,1.525941,3.287982,-3.474332,...,-3.571927,NaN,7.481317,-0.994704,-18.834546,NaN,10.528915,-8.060978,-3.586118,NaN
4,2016Q1,-8.612440,-8.230453,-2.800000,0.000000,1.526718,0.577471,-0.501002,0.109769,-19.863079,...,-8.066513,-3.571927,29.439090,7.481317,-0.994704,-18.834546,-3.447354,10.528915,-8.060978,-3.586118
5,2016Q2,10.471204,6.278027,1.646091,-3.529412,-1.503759,0.510445,2.014099,1.425439,20.102284,...,10.531486,-8.066513,-12.704340,29.439090,7.481317,-0.994704,-19.855212,-3.447354,10.528915,-8.060978
6,2016Q3,0.000000,-2.531646,-1.214575,4.065041,0.381679,-0.128542,-0.789733,-1.513514,20.635901,...,-3.474332,10.531486,-6.258746,-12.704340,29.439090,7.481317,20.070732,-19.855212,-3.447354,10.528915
7,2016Q4,1.895735,6.493506,2.049180,-2.343750,1.901141,0.883248,0.597015,2.195390,8.364802,...,-19.863079,-3.474332,4.432232,-6.258746,-12.704340,29.439090,20.620760,20.070732,-19.855212,-3.447354
8,2017Q1,6.046512,5.691057,1.606426,0.400000,-0.746269,-0.740917,-0.989120,-0.107411,-13.430523,...,20.102284,-19.863079,17.397561,4.432232,-6.258746,-12.704340,8.372109,20.620760,20.070732,-19.855212
9,2017Q2,-0.877193,1.153846,0.790514,1.992032,-5.263158,2.502468,1.198801,0.537634,-3.543691,...,20.635901,20.102284,-14.378691,17.397561,4.432232,-6.258746,-13.431354,8.372109,20.620760,20.070732


In [48]:
# 숫자형 컬럼만 선택 (연도분기 제외)
numeric_df = df2.select_dtypes(include=["number"])

# 전체 상관계수
corr = numeric_df.corr()

target_corr = corr["gdp"].drop("gdp")
target_corr = target_corr.sort_values(ascending=False)
target_corr

,gdp
생산지수,0.446011
인건비_lag4,0.398273
자금사정실적BSI_lag4,0.307693
업황실적BSI,0.306582
소매판매액지수_lag4,0.306264
매출실적BSI,0.291377
생산지수_lag4,0.279582
채산성실적BSI_lag3,0.278718
업황실적BSI_lag4,0.276083
인력사정실적BSI_lag3,0.269025


In [49]:
target_corr.to_csv('도소매업순서.csv')

In [50]:
df2.dropna(inplace=True)

In [51]:
df2

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,gdp,생산지수,소매판매액지수,EBITDA,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
4,2016Q1,-8.612440,-8.230453,-2.800000,0.000000,1.526718,0.577471,-0.501002,0.109769,-19.863079,...,-8.066513,-3.571927,29.439090,7.481317,-0.994704,-18.834546,-3.447354,10.528915,-8.060978,-3.586118
5,2016Q2,10.471204,6.278027,1.646091,-3.529412,-1.503759,0.510445,2.014099,1.425439,20.102284,...,10.531486,-8.066513,-12.704340,29.439090,7.481317,-0.994704,-19.855212,-3.447354,10.528915,-8.060978
6,2016Q3,0.000000,-2.531646,-1.214575,4.065041,0.381679,-0.128542,-0.789733,-1.513514,20.635901,...,-3.474332,10.531486,-6.258746,-12.704340,29.439090,7.481317,20.070732,-19.855212,-3.447354,10.528915
7,2016Q4,1.895735,6.493506,2.049180,-2.343750,1.901141,0.883248,0.597015,2.195390,8.364802,...,-19.863079,-3.474332,4.432232,-6.258746,-12.704340,29.439090,20.620760,20.070732,-19.855212,-3.447354
8,2017Q1,6.046512,5.691057,1.606426,0.400000,-0.746269,-0.740917,-0.989120,-0.107411,-13.430523,...,20.102284,-19.863079,17.397561,4.432232,-6.258746,-12.704340,8.372109,20.620760,20.070732,-19.855212
9,2017Q2,-0.877193,1.153846,0.790514,1.992032,-5.263158,2.502468,1.198801,0.537634,-3.543691,...,20.635901,20.102284,-14.378691,17.397561,4.432232,-6.258746,-13.431354,8.372109,20.620760,20.070732
10,2017Q3,3.097345,-0.380228,0.392157,-3.906250,1.190476,1.666096,0.394867,0.320856,15.103626,...,8.364802,20.635901,-1.537073,-14.378691,17.397561,4.432232,-3.541952,-13.431354,8.372109,20.620760
11,2017Q4,1.287554,-1.908397,-2.343750,5.284553,2.352941,0.225355,1.179941,2.132196,4.579011,...,-13.430523,8.364802,-0.910547,-1.537073,-14.378691,17.397561,15.089456,-3.541952,-13.431354,8.372109
12,2018Q1,5.508475,4.280156,7.600000,-0.386100,1.149425,-1.004081,0.388727,2.400835,-2.413082,...,-3.543691,-13.430523,103.360843,-0.910547,-1.537073,-14.378691,4.654265,15.089456,-3.541952,-13.431354
13,2018Q2,1.204819,-1.119403,-2.602230,0.775194,-5.303030,1.724471,0.193611,-0.101937,-7.097062,...,15.103626,-3.543691,-44.241030,103.360843,-0.910547,-1.537073,-2.475002,4.654265,15.089456,-3.541952


In [52]:
df2 = df2.reset_index()
df2 = df2.drop(columns=["level_0"])

In [53]:
df2.to_csv('도소매업lag.csv', encoding='utf-8-sig')

#ARIMA

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('도소매업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','인건비_lag4','자금사정실적BSI_lag4','업황실적BSI','소매판매액지수_lag4','인력사정실적BSI_lag1']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 0.7736
📈 평균 Test MAE: 1.3814


## 아리마 예측값

In [54]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('도소매업lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','인건비_lag4','자금사정실적BSI_lag4','업황실적BSI','소매판매액지수_lag4','인력사정실적BSI_lag1']

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    pred = fit.forecast(steps=1, exog=X_test)

                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 최적 (p,d,q) 선택
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

best_p, best_d, best_q = int(best['p']), int(best['d']), int(best['q'])

print("✅ 최적 ARIMAX(p,d,q):", (best_p, best_d, best_q))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

# ---------------------------------------------------
# 8️⃣ 최적 (p,d,q)로 전체 구간 롤링 예측값 저장
# ---------------------------------------------------
rolling_preds = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(train_data) < train_window or len(X_test) == 0:
        continue

    try:
        model = SARIMAX(train_data, exog=X_train, order=(best_p, best_d, best_q),
                        enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False)
        pred = fit.forecast(steps=1, exog=X_test)

        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(y.loc[test_end])
        })
    except:
        continue

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 최적 모델 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 ARIMAX(p,d,q): (0, 1, 1)
📉 평균 Train MAE: 0.7736
📈 평균 Test MAE: 1.3814

📌 최적 모델 예측값(pred) 시계열:
            pred    actual
date                      
2021Q1 -0.332416 -1.638957
2021Q2  1.922988  1.320764
2021Q3  2.346630 -0.449570
2021Q4  1.307825  2.849366
2022Q1 -0.666141 -1.181820
2022Q2  1.212677  2.580375
2022Q3  1.691954  0.643921
2022Q4  0.916690  1.512812
2023Q1 -0.920230 -2.230239
2023Q2 -0.118055 -2.575493
2023Q3 -1.525490 -1.877107
2023Q4 -1.886450 -0.974690
2024Q1 -0.604999  1.959271
2024Q2  2.204747  0.337139
2024Q3 -0.382652 -1.998314
2024Q4 -2.259294 -1.210548
2025Q1 -2.071858  0.448430
2025Q2  4.197274  3.753878


In [56]:
level_2025Q1 = 42806.4

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 44603.102027066816


#랜덤포레스트

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','인건비_lag4','자금사정실적BSI_lag4','업황실적BSI','소매판매액지수_lag4','인력사정실적BSI_lag1']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

# 첫 번째 윈도우 구간 설정
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# 하이퍼파라미터 탐색 (단 1회)
rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,            # 108조합 중 10개만 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 0.5579
📈 평균 Test MAE: 1.5265


## 랜포 예측값

In [57]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','인건비_lag4','자금사정실적BSI_lag4','업황실적BSI','소매판매액지수_lag4','인력사정실적BSI_lag1'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ RandomizedSearchCV (초기 구간 1회)
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []   # ⭐ 예측값 저장 리스트

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred),
            'actual': float(y_test.values[0])
        })

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 정리
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 랜덤포레스트 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 0.5579
📈 평균 Test MAE: 1.5265

📌 랜덤포레스트 예측값(pred) 시계열:
            pred    actual
date                      
2021Q1 -0.010479 -1.638957
2021Q2  0.511819  1.320764
2021Q3  0.782173 -0.449570
2021Q4  1.184298  2.849366
2022Q1 -0.227889 -1.181820
2022Q2  1.397902  2.580375
2022Q3  1.143820  0.643921
2022Q4  0.522726  1.512812
2023Q1 -0.505679 -2.230239
2023Q2  0.036227 -2.575493
2023Q3  0.409758 -1.877107
2023Q4 -0.624177 -0.974690
2024Q1 -0.554399  1.959271
2024Q2  1.015005  0.337139
2024Q3 -0.481833 -1.998314
2024Q4 -0.598877 -1.210548
2025Q1 -1.681881  0.448430
2025Q2 -0.339077  3.753878


In [58]:
level_2025Q1 = 42806.4

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 42661.25317155768


#AR

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업qoq.csv')

# GDP 단일 시계열만 사용 (AR(1))
y = df['gdp'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train 예측 ---
        # fittedvalues는 길이가 n-1이므로 첫 번째 값 제외하고 MAE 계산
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 (1분기 앞 예측) ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 1.1851
📈 평균 Test MAE: 1.7966


## AR 예측값

In [63]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업qoq.csv')

# GDP 단일 시계열
y = df['gdp'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# ⭐ 예측값 저장할 리스트
rolling_preds = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train MAE ---
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(test_data.values[0])
        })

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

# -----------------------------
# 5️⃣ pred 시계열 데이터프레임 출력
# -----------------------------
pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 AR(1) 예측값(pred) 시계열:")
print(pred_df)


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 1.1851
📈 평균 Test MAE: 1.7966

📌 AR(1) 예측값(pred) 시계열:
            pred    actual
date                      
2020Q1  0.747863 -1.923134
2020Q2  1.196245 -1.515502
2020Q3  0.147048  1.941871
2020Q4  0.529383  1.475371
2021Q1  0.630492 -1.638957
2021Q2  0.551696  1.320764
2021Q3  0.478674 -0.449570
2021Q4  0.586227  2.849366
2022Q1  0.301536 -1.181820
2022Q2  0.814827  2.580375
2022Q3 -0.228350  0.643921
2022Q4  0.509909  1.512812
2023Q1  0.365313 -2.230239
2023Q2  1.522227 -2.575493
2023Q3  0.664769 -1.877107
2023Q4  0.116848 -0.974690
2024Q1  0.006569  1.959271
2024Q2  0.071939  0.337139
2024Q3  0.057548 -1.998314
2024Q4 -0.021519 -1.210548
2025Q1 -0.092632  0.448430
2025Q2  0.037844  3.753878


In [64]:
level_2025Q1 = 42806.4

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 42822.599581614115


#XGB

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','인건비_lag4','자금사정실적BSI_lag4','업황실적BSI','소매판매액지수_lag4','인력사정실적BSI_lag1'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

# 첫 번째 윈도우로 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# RandomizedSearchCV 실행
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,          # 전체 중 15조합 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 600, 'min_child_weight': 1, 'max_depth': 9, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.8}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.1314
📈 평균 Test MAE: 1.6802


##XGB 예측값

In [65]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = 'gdp'
exog_vars = [
'생산지수','인건비_lag4','자금사정실적BSI_lag4','업황실적BSI','소매판매액지수_lag4','인력사정실적BSI_lag1'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ RandomizedSearchCV (초기 구간)
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []   # ⭐ 예측값 저장

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred),
            'actual': float(y_test.values[0])
        })

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 XGBoost 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 600, 'min_child_weight': 1, 'max_depth': 9, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.8}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.1314
📈 평균 Test MAE: 1.6802

📌 XGBoost 예측값(pred) 시계열:
            pred    actual
date                      
2021Q1 -0.025251 -1.638957
2021Q2  0.765306  1.320764
2021Q3  0.803832 -0.449570
2021Q4  0.819296  2.849366
2022Q1 -0.596561 -1.181820
2022Q2  1.534723  2.580375
2022Q3  1.017915  0.643921
2022Q4  0.040301  1.512812
2023Q1 -0.863062 -2.230239
2023Q2  0.223717 -2.575493
2023Q3  0.293190 -1.877107
2023Q4 -0.567692 -0.974690
2024Q1 -1.775995  1.959271
2024Q2  1.924062  0.337139
2024Q3 -1.144123 -1.998314
2024Q4 -0.476671 -1.210548
2025Q1 -2.353349  0.448430
2025Q2 -1.104258  3.753878


In [66]:
level_2025Q1 = 42806.4

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 42333.706979667666


#MLP

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','인건비_lag4','자금사정실적BSI_lag4','업황실적BSI','소매판매액지수_lag4','인력사정실적BSI_lag1'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# 스케일링 (NN에서는 매우 중요)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16), (64,32), (128,64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

# 첫 번째 윈도우로 하이퍼파라미터 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=1000)
rand_search = RandomizedSearchCV(
    mlp,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y_scaled.loc[train_start:train_end]
    y_test = y_scaled.loc[test_end:test_end]
    X_train = X_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = MLPRegressor(**best_params, random_state=42, max_iter=1000)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred.reshape(-1, 1)).flatten()
        test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1)).flatten()
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1)).flatten()

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (32,), 'alpha': 0.01, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.1749
📈 평균 Test MAE: 1.7618


## MLP 예측값

In [67]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = 'gdp'
exog_vars = [
'생산지수','인건비_lag4','자금사정실적BSI_lag4','업황실적BSI','소매판매액지수_lag4','인력사정실적BSI_lag1'
]

df_model = df[[target] + exog_vars].dropna()

y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링 (정상 코드)
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(
    scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(),
    index=y.index
)

# -----------------------------
# 4️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 5️⃣ 하이퍼파라미터 축소 (안정성)
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16)],
    'activation': ['relu'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.005]
}

first_train_end = start_period + (train_window - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=2000)
rand_search = RandomizedSearchCV(
    mlp, param_grid, n_iter=5, cv=3,
    scoring='neg_mean_absolute_error', n_jobs=-1, random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 파라미터:", best_params)

# -----------------------------
# 6️⃣ 롤링 예측
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    train_start = train_end - (train_window - 1)
    test_end = test_start

    X_train = X_scaled.loc[train_start:train_end]
    y_train = y_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]
    y_test = y_scaled.loc[test_end:test_end]

    if len(X_train) < train_window:
        continue

    try:
        model = MLPRegressor(
            **best_params,
            random_state=42,
            max_iter=2000
        )
        model.fit(X_train, y_train)

        # ---- Train 예측 ----
        train_pred_scaled = model.predict(X_train)
        train_pred = scaler_y.inverse_transform(train_pred_scaled.reshape(-1,1)).flatten()
        train_real = scaler_y.inverse_transform(y_train.values.reshape(-1,1)).flatten()
        train_mae = mean_absolute_error(train_real, train_pred)
        train_mae_list.append(train_mae)

        # ---- Test 예측 ----
        test_pred_scaled = model.predict(X_test)
        test_pred = scaler_y.inverse_transform(test_pred_scaled.reshape(-1,1)).flatten()
        test_real = scaler_y.inverse_transform(y_test.values.reshape(-1,1)).flatten()
        test_mae = mean_absolute_error(test_real, test_pred)
        test_mae_list.append(test_mae)

        # ---- pred 저장 ----
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred[0]),
            'actual': float(test_real[0])
        })

    except Exception as e:
        print("❌ 오류 발생:", e)
        continue

# -----------------------------
# 7️⃣ 결과 출력
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

if len(rolling_preds) > 0:
    pred_df = pd.DataFrame(rolling_preds).set_index('date')
    print(pred_df)
else:
    print("⚠ rolling_preds가 텅 빔 → 모델 학습 실패")


✅ 최적 파라미터: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (32,), 'alpha': 0.0001, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.1653
📈 평균 Test MAE: 1.7673
            pred    actual
date                      
2021Q1  1.406038 -1.638957
2021Q2  1.477814  1.320764
2021Q3  1.345829 -0.449570
2021Q4  1.126063  2.849366
2022Q1 -0.307753 -1.181820
2022Q2  1.353284  2.580375
2022Q3  2.270999  0.643921
2022Q4  0.335097  1.512812
2023Q1 -1.598199 -2.230239
2023Q2  0.068052 -2.575493
2023Q3  0.487966 -1.877107
2023Q4  0.834103 -0.974690
2024Q1  0.555066  1.959271
2024Q2  2.204219  0.337139
2024Q3 -0.567292 -1.998314
2024Q4 -0.066972 -1.210548
2025Q1 -2.220557  0.448430
2025Q2 -0.466646  3.753878


In [68]:
level_2025Q1 = 42806.4

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 42606.645457390834


#LSTM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','인건비_lag4','자금사정실적BSI_lag4','업황실적BSI','소매판매액지수_lag4','인력사정실적BSI_lag1']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 4️⃣ LSTM 입력 형태 변환 함수
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    """
    lookback: 과거 몇 분기 데이터를 사용할지 (ex. 4 = 1년)
    """
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ LSTM 모델 정의 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측
# -----------------------------
train_mae_list, test_mae_list = [], []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    # 윈도우 범위 맞추기
    train_mask = (np.array(idxs) >= train_start) & (np.array(idxs) <= train_end)
    test_mask = (np.array(idxs) == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test, y_test = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
        model.fit(X_train, y_train, epochs=200, batch_size=8, verbose=0, callbacks=[early_stop])

        # 예측
        train_pred = model.predict(X_train, verbose=0)
        test_pred = model.predict(X_test, verbose=0)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1))
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1))

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 8️⃣ 결과 요약
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 0.3381
📈 평균 Test MAE: 1.7374


## LSTM 예측값

In [69]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = 'gdp'
exog_vars = [
'생산지수','인건비_lag4','자금사정실적BSI_lag4','업황실적BSI','소매판매액지수_lag4','인력사정실적BSI_lag1'
]

df_model = pd.concat([df[target], df[exog_vars]], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(
    scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(),
    index=y.index
)

# -----------------------------
# 4️⃣ LSTM 입력 생성
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ 모델 생성 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list, test_mae_list = [], []
rolling_preds = []  # ⭐ 예측값 저장 리스트

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    train_start = train_end - (train_window - 1)
    test_end = test_start

    # LSTM용 슬라이싱
    idx_arr = np.array(idxs)

    train_mask = (idx_arr >= train_start) & (idx_arr <= train_end)
    test_mask  = (idx_arr == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test,  y_test  = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)

        model.fit(X_train, y_train, epochs=200, batch_size=8,
                  verbose=0, callbacks=[early])

        # ---- Train 예측 ----
        train_pred = model.predict(X_train, verbose=0)
        test_pred  = model.predict(X_test, verbose=0)

        # ---- 스케일 복원 ----
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1,1))

        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_test_inv  = scaler_y.inverse_transform(y_test.values.reshape(-1,1))

        # ---- MAE ----
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred_inv[0]),
            'actual': float(y_test_inv[0])
        })

    except Exception as e:
        print(f"❌ Error: {e}")
        continue

# -----------------------------
# 8️⃣ 결과 요약 + pred_df 출력
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE:  {np.mean(test_mae_list):.4f}")

# ⭐ 예측값 시계열 출력
pred_df = pd.DataFrame(rolling_preds).set_index('date')
print("\n📌 LSTM 예측값(pred) 시계열:")
print(pred_df)



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 0.3136
📈 평균 Test MAE:  1.9079

📌 LSTM 예측값(pred) 시계열:
            pred    actual
date                      
2022Q1  0.894823 -1.181820
2022Q2  1.839344  2.580375
2022Q3  1.983948  0.643921
2022Q4  0.867397  1.512812
2023Q1 -0.010851 -2.230239
2023Q2  1.574314 -2.575493
2023Q3 -0.066505 -1.877107
2023Q4 -0.450808 -0.974690
2024Q1 -1.028291  1.959271
2024Q2  0.918435  0.337139
2024Q3  0.491915 -1.998314
2024Q4 -1.840047 -1.210548
2025Q1 -1.955540  0.448430
2025Q2 -0.357687  3.753878


In [70]:
level_2025Q1 = 42806.4

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 42653.28713733387


#단변량 아리마

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업lag.csv')

# 인덱스가 분기형일 경우 변환
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상 설정
# -----------------------------
y = df['gdp'].dropna()

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간
start_period = pd.Period('2021Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ (p,d,q) 탐색 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                # 데이터 슬라이싱
                y_train = y.loc[train_start:train_end]
                y_test = y.loc[test_end:test_end]

                if len(y_train) < train_window:
                    continue

                try:
                    # -------------------------
                    # 모델 학습
                    # -------------------------
                    model = ARIMA(y_train, order=(p, d, q))
                    fit = model.fit()

                    # -------------------------
                    # 예측 및 MAE 계산
                    # -------------------------
                    pred = fit.forecast(steps=1)

                    train_mae = mean_absolute_error(y_train, fit.fittedvalues)
                    test_mae = mean_absolute_error(y_test, pred)

                    train_mae_list.append(train_mae)
                    test_mae_list.append(test_mae)

                except Exception as e:
                    print(f"❌ Error at (p,d,q)=({p},{d},{q}): {e}")
                    continue

            # -------------------------
            # 평균 MAE 저장
            # -------------------------
            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    print("⚠️ 결과가 없습니다. 모든 (p,d,q) 조합이 실패했습니다.")
else:
    best = results_df.loc[results_df['test_MAE'].idxmin()]
    print("✅ 최적 ARIMA(p,d,q):", tuple(best[['p', 'd', 'q']]))
    print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
    print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

    display(results_df.sort_values('test_MAE').head(10))


✅ 최적 ARIMA(p,d,q): (3.0, 1.0, 0.0)
📉 평균 Train MAE: 1.2712
📈 평균 Test MAE: 1.6559


,p,d,q,train_MAE,test_MAE
36,3,1,0,1.271172,1.655950
41,3,2,1,1.500033,1.682419
1,0,1,1,1.349135,1.697738
13,1,1,1,1.328272,1.752029
2,0,1,2,1.320066,1.755703
19,1,2,3,1.534262,1.774800
14,1,1,2,1.292290,1.790984
27,2,1,3,1.198283,1.799967
42,3,2,2,1.478534,1.841775
38,3,1,2,1.191260,1.849979


## 단변량 아리마 예측

In [71]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('도소매업lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상
# -----------------------------
y = df['gdp'].dropna()

# -----------------------------
# 3️⃣ 롤링 윈도우
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2021Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ (p,d,q) 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                y_train = y.loc[train_start:train_end]
                y_test = y.loc[test_end:test_end]

                if len(y_train) < train_window:
                    continue

                try:
                    model = ARIMA(y_train, order=(p, d, q))
                    fit = model.fit()

                    pred = fit.forecast(steps=1)

                    train_mae = mean_absolute_error(y_train, fit.fittedvalues)
                    test_mae = mean_absolute_error(y_test, pred)

                    train_mae_list.append(train_mae)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 최적 모델 선택
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    print("⚠️ 모든 조합 실패")
else:
    best = results_df.loc[results_df['test_MAE'].idxmin()]
    best_p, best_d, best_q = int(best['p']), int(best['d']), int(best['q'])

    print("✅ 최적 ARIMA(p,d,q):", (best_p, best_d, best_q))
    print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
    print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

# -----------------------------
# 8️⃣ ⭐ 최적 모델로 pred 저장 (재예측)
# -----------------------------
rolling_preds = []
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]

    if len(y_train) < train_window:
        continue

    try:
        model = ARIMA(y_train, order=(best_p, best_d, best_q))
        fit = model.fit()

        pred = fit.forecast(steps=1)

        # MAE
        train_mae_list.append(mean_absolute_error(y_train, fit.fittedvalues))
        test_mae_list.append(mean_absolute_error(y_test, pred))

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(y_test.values[0])
        })

    except Exception as e:
        print("❌ 재예측 오류:", e)
        continue

# -----------------------------
# 9️⃣ pred_df 출력
# -----------------------------
pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 ARIMA 최적모델 예측(pred) 시계열:")
print(pred_df)

print("\n📉 최종 Train MAE:", np.mean(train_mae_list))
print("📈 최종 Test MAE:", np.mean(test_mae_list))


✅ 최적 ARIMA(p,d,q): (3, 1, 0)
📉 평균 Train MAE: 1.2712
📈 평균 Test MAE: 1.6559

📌 ARIMA 최적모델 예측(pred) 시계열:
            pred    actual
date                      
2021Q1 -0.772507 -1.638957
2021Q2 -1.459197  1.320764
2021Q3  1.595276 -0.449570
2021Q4  0.465035  2.849366
2022Q1 -0.685009 -1.181820
2022Q2  1.181454  2.580375
2022Q3 -0.182143  0.643921
2022Q4  2.285752  1.512812
2023Q1 -0.086868 -2.230239
2023Q2  1.826241 -2.575493
2023Q3 -1.134559 -1.877107
2023Q4 -0.518062 -0.974690
2024Q1 -1.664654  1.959271
2024Q2 -0.312409  0.337139
2024Q3 -0.082143 -1.998314
2024Q4 -1.495367 -1.210548
2025Q1 -0.120905  0.448430
2025Q2  0.305181  3.753878

📉 최종 Train MAE: 1.2711717405766496
📈 최종 Test MAE: 1.6559499875895842


In [72]:
level_2025Q1 = 42806.4

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 42937.036899213235
